In [1]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
df = pd.read_csv("Youtube-KatyPerry.csv")

print("Dataset loaded successfully")
print(df.head())

Dataset loaded successfully
                              COMMENT_ID        AUTHOR                 DATE  \
0      z12pgdhovmrktzm3i23es5d5junftft3f   lekanaVEVO1  2014-07-22T15:27:50   
1    z13yx345uxepetggz04ci5rjcxeohzlrtf4      Pyunghee  2014-07-27T01:57:16   
2  z12lsjvi3wa5x1vwh04cibeaqnzrevxajw00k    Erica Ross  2014-07-27T02:51:43   
3    z13jcjuovxbwfr0ge04cev2ipsjdfdurwck  Aviel Haimov  2014-08-01T12:27:48   
4  z13qybua2yfydzxzj04cgfpqdt2syfx53ms0k    John Bello  2014-08-01T21:04:03   

                                             CONTENT  CLASS  
0  i love this so much. AND also I Generate Free ...      1  
1  http://www.billboard.com/articles/columns/pop-...      1  
2  Hey guys! Please join me in my fight to help a...      1  
3  http://psnboss.com/?ref=2tGgp3pV6L this is the...      1  
4  Hey everyone. Watch this trailer!!!!!!!!  http...      1  


In [5]:
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nClass distribution:")
print(df["CLASS"].value_counts())

Number of rows: 350
Columns: ['COMMENT_ID', 'AUTHOR', 'DATE', 'CONTENT', 'CLASS']

Class distribution:
CLASS
1    175
0    175
Name: count, dtype: int64


In [6]:
def clean_comment(comment):
    comment = str(comment).lower()
    comment = re.sub(r"http\S+|www\S+", "", comment)
    comment = re.sub(r"[^a-z\s]", "", comment)
    comment = re.sub(r"\s+", " ", comment)
    
    return comment.strip()

df["clean_text"] = df["CONTENT"].apply(clean_comment)

print(df[["CONTENT", "clean_text"]].head())

                                             CONTENT  \
0  i love this so much. AND also I Generate Free ...   
1  http://www.billboard.com/articles/columns/pop-...   
2  Hey guys! Please join me in my fight to help a...   
3  http://psnboss.com/?ref=2tGgp3pV6L this is the...   
4  Hey everyone. Watch this trailer!!!!!!!!  http...   

                                          clean_text  
0  i love this so much and also i generate free l...  
1  vote for sones pleasewere against vipsplease h...  
2  hey guys please join me in my fight to help ab...  
3                                   this is the song  
4                    hey everyone watch this trailer  


In [7]:
text_data = df["clean_text"]
labels = df["CLASS"]

print("Text samples:", len(text_data))
print("Labels:", len(labels))

Text samples: 350
Labels: 350


In [8]:
text_train, text_test, label_train, label_test = train_test_split(
    text_data,
    labels,
    test_size=0.25,
    random_state=7,
    stratify=labels
)

print("Training records:", len(text_train))
print("Testing records:", len(text_test))

Training records: 262
Testing records: 88


In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=1
)

train_matrix = vectorizer.fit_transform(text_train)
test_matrix = vectorizer.transform(text_test)

print("TF-IDF matrix:", train_matrix.shape)

TF-IDF matrix: (262, 910)


In [10]:
classifier = MultinomialNB()

classifier.fit(train_matrix, label_train)

print("Naive Bayes model trained")

Naive Bayes model trained


In [11]:
predicted_labels = classifier.predict(test_matrix)

print("First 10 predictions:")
print(predicted_labels[:10])

First 10 predictions:
[0 0 0 0 0 0 1 0 0 0]


In [12]:
accuracy = accuracy_score(label_test, predicted_labels)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(label_test, predicted_labels))

print("\nConfusion Matrix:")
print(confusion_matrix(label_test, predicted_labels))

Accuracy: 76.14 %

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.98      0.80        44
           1       0.96      0.55      0.70        44

    accuracy                           0.76        88
   macro avg       0.82      0.76      0.75        88
weighted avg       0.82      0.76      0.75        88


Confusion Matrix:
[[43  1]
 [20 24]]


In [13]:
test_comments = [
    "Amazing video, I really enjoyed it",
    "Click this link and get a free prize",
    "This song is really good",
    "Congratulations you have won money",
    "Thank you for sharing this video"
]

prepared_comments = [clean_comment(x) for x in test_comments]

comment_vectors = vectorizer.transform(prepared_comments)

predictions = classifier.predict(comment_vectors)

for comment, prediction in zip(test_comments, predictions):
    if prediction == 1:
        print("SPAM   :", comment)
    else:
        print("NORMAL :", comment)

NORMAL : Amazing video, I really enjoyed it
SPAM   : Click this link and get a free prize
NORMAL : This song is really good
SPAM   : Congratulations you have won money
SPAM   : Thank you for sharing this video
